In [1]:
import os
import zipfile
import requests
import tempfile
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from adabord._adabord import ADABORD
from adabord._metrics import amae
from sklearn.metrics import balanced_accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold

In [2]:
url = "https://www.uco.es/grupos/ayrna/datasets/TOC-UCO.zip"
response = requests.get(url)

temp_file = tempfile.NamedTemporaryFile(delete=False)
temp_file.write(response.content)

tmp_tocuco_path = tempfile.mkdtemp()
zip_ref = zipfile.ZipFile(temp_file.name, 'r')
zip_ref.extractall(tmp_tocuco_path)
extracted_files = zip_ref.namelist()

tmp_tocuco_path = os.path.join(tmp_tocuco_path, "TOC-UCO")

print(f"TOC-UCO repository saved temporarily at {tmp_tocuco_path}")
print(f"Files inside the repository: {os.listdir(tmp_tocuco_path)}")

TOC-UCO repository saved temporarily at /tmp/tmptju5s8_h/TOC-UCO
Files inside the repository: ['train_masks.pkl', 'metadata.csv', 'data', 'train_masks.json']


In [3]:
SEEDS = 2 #! Seeds should be <= 30, in the paper we consider 30 seeds

## Get training masks to make the train/test partitions
with open(os.path.join(tmp_tocuco_path, "train_masks.pkl"), "rb") as train_masks_binary:
    train_masks = joblib.load(train_masks_binary)

tocuco_datasets_path = os.path.join(tmp_tocuco_path, "data")

# initialize cv parameters
amae_scorer = make_scorer(amae, greater_is_better=False)
random_search_n_iters = 2 #! In the paper we consider 20 iterations
random_search_cv_method = StratifiedKFold(n_splits=3)

for dataset_name in os.listdir(tocuco_datasets_path):
    dataset = pd.read_csv(os.path.join(tocuco_datasets_path, dataset_name))
    for seed in range(SEEDS):
        dataset_name_without_extension = dataset_name.split(".")[0]

        # Get train mask for the current dataset and seed
        dataset_seed_train_mask = train_masks[f"{dataset_name_without_extension}_seed_{seed}"]

        # Apply the mask to get train and test partitions
        train = dataset.loc[dataset_seed_train_mask]
        test = dataset.loc[~dataset_seed_train_mask]

        # Separate features and target variable and convert to numpy arrays
        X_train = train.drop(columns=["y"]).to_numpy()
        X_test = test.drop(columns=["y"]).to_numpy()
        y_train = train["y"].to_numpy()
        y_test = test["y"].to_numpy()
        del train, test

        adabord_param_grid = {
            "n_estimators": [10, 20], #! In the paper we consider [50, 100, 250, 500, 1000, 2000]
            "estimator": [
                DecisionTreeClassifier(max_depth=2, criterion="ogini", random_state=seed),
                DecisionTreeClassifier(max_depth=4, criterion="ogini", random_state=seed),
                # DecisionTreeClassifier(max_depth=8, criterion="ogini", random_state=seed),
                # DecisionTreeClassifier(max_depth=16, criterion="ogini", random_state=seed),
                # DecisionTreeClassifier(max_depth=None, criterion="ogini", random_state=seed),
            ],
        }

        adabord_cv = RandomizedSearchCV(
            ADABORD(random_state=seed),
            scoring=amae_scorer,
            param_distributions=adabord_param_grid,
            n_iter=random_search_n_iters,
            cv=random_search_cv_method,
            n_jobs=1,
            verbose=3,
            random_state=seed,
            error_score="raise",
        )

        adabord_cv.fit(X_train, y_train)
        best_adabord = adabord_cv.best_estimator_
        y_pred = best_adabord.predict(X_test)
        balanced_acc = balanced_accuracy_score(y_test, y_pred)
        print(f"Dataset: {dataset_name_without_extension}, Seed: {seed}, Balanced Accuracy: {balanced_acc}, AMAE: {amae(y_test, y_pred)}")

Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV 1/3] END estimator=DecisionTreeClassifier(criterion='ogini', max_depth=4, random_state=0), n_estimators=10;, score=-1.172 total time=   0.1s
[CV 2/3] END estimator=DecisionTreeClassifier(criterion='ogini', max_depth=4, random_state=0), n_estimators=10;, score=-1.211 total time=   0.1s
[CV 3/3] END estimator=DecisionTreeClassifier(criterion='ogini', max_depth=4, random_state=0), n_estimators=10;, score=-1.112 total time=   0.1s
[CV 1/3] END estimator=DecisionTreeClassifier(criterion='ogini', max_depth=4, random_state=0), n_estimators=20;, score=-1.132 total time=   0.2s
[CV 2/3] END estimator=DecisionTreeClassifier(criterion='ogini', max_depth=4, random_state=0), n_estimators=20;, score=-1.159 total time=   0.2s
[CV 3/3] END estimator=DecisionTreeClassifier(criterion='ogini', max_depth=4, random_state=0), n_estimators=20;, score=-1.074 total time=   0.2s
Dataset: oc04_LEVXSensors, Seed: 0, Balanced Accuracy: 0.37115762693174